# GT-04k — Équilibre corrélé (Aumann 1974) : le médiateur bat le Nash

**Navigation** : [<< 4j-Rationalisabilite](GameTheory-04j-Rationalisabilite-Python.ipynb) | [Index](README.md) | [05-ZeroSum-Minimax >>](GameTheory-05-ZeroSum-Minimax-Python.ipynb)

**Série GameTheory** — famille 04 (Nash / formes normales).

Dans [GT-04](GameTheory-04-NashEquilibrium-Python.ipynb) on a vu que les équilibres de
Nash d'un jeu de coordination peuvent être **inequitables** : les joueurs se coordonnent
sur l'équilibre « mauvais » et n'osent pas s'en écarter, car aucun ne gagne à changer
**unilatéralement**. L'équilibre **corrélé** (Aumann, 1974) brise ce verrou : un
**médiateur** commun tire un profil d'actions selon une distribution μ et recommande
*en privé* à chaque joueur quelle action jouer. Si, sachant sa recommandation, aucun
joueur ne gagne à dévier, μ est un équilibre corrélé.

L'idée clé, que ce carnet **mesure** sur un jeu 2×2 :

- l'ensemble des équilibres corrélés est un **polytope** — on le calcule en LP ;
- la corrélation permet d'atteindre un payoff que **aucun** Nash n'atteint ;
- le Nash est un cas particulier du corrélé (les NE sont des CE « dégénérés »).

**Objectifs pédagogiques**

1. Écrire les **quatre** inégalités de non-déviabilité d'un jeu 2×2, et les résoudre par
   programmation linéaire (`scipy.optimize.linprog`).
2. Comparer, **sur le même jeu**, le payoff du médiateur et celui du Nash mixte : l'écart
   ne tient pas à une différence de moyens, mais à une **recommandation coordonnée**.
3. Distinguer *être corrélé* (recevoir une recommandation commune) de *satisfaire les
   inégalités de non-déviabilité* — le carnet oppose un **contre-exemple** mesuré.


## 1. Le jeu : Battle of the Sexes

Deux joueurs, deux actions chacune. La première (M) préfère l'Opéra, le second (J)
préfère le Football ; mais ils préfèrent **se retrouver ensemble** plutôt que de se
séparer.

|            | J: F | J: O |
|------------|------|------|
| **M: F**   | 3, 1 | 0, 0 |
| **M: O**   | 0, 0 | 1, 3 |

Payoffs (M, J). Les deux équilibres de Nash purs sont **(F,F)** → (3,1) et
**(O,O)** → (1,3) : coordonnés, mais *inéquitable* — l'un des deux « perd ».


In [1]:
import numpy as np

# Battle of the Sexes — payoffs (M, J), actions [F, O]
# index 0 = F, index 1 = O
A = np.array([[3, 0],
              [0, 1]])   # payoff de M  (lignes = action M, colonnes = action J)
B = np.array([[1, 0],
              [0, 3]])   # payoff de J
actions = ["F", "O"]
print("Payoff M (lignes=M, colonnes=J):")
print(A)
print("Payoff J:")
print(B)


Payoff M (lignes=M, colonnes=J):
[[3 0]
 [0 1]]
Payoff J:
[[1 0]
 [0 3]]


### Lecture du résultat — la table des gains et ses deux coins

`A[i,j]` est le payoff de **M** quand M joue la ligne `i` et J la colonne `j` ;
`B[i,j]` celui de **J** sur le même couple d'actions. La sortie de la cellule
précédente affiche `A = [[3, 0], [0, 1]]` et `B = [[1, 0], [0, 3]]` : les deux coins de
la diagonale portent les deux gains **positifs** — (F,F) donne (3, 1) et (O,O) donne
(1, 3) — et les deux cases hors diagonale portent **zéro** pour les deux joueurs.

C'est un jeu de **coordination** : les deux joueurs veulent se retrouver *ensemble*, mais
pas sur la même case. M préfère (F,F) (3 plutôt que 1), J préfère (O,O) (3 plutôt
que 1). La coordination a donc un prix : sur les deux équilibres purs, l'un des deux
joueurs obtient 3 et l'autre 1 — ils sont **inequitables** l'un pour l'autre. C'est le
point de départ du carnet : le §2 cherche une issue qui rende les deux joueurs
**aussi bien servis**, et le §3 mesure d'abord ce que le Nash mixte propose de son
côté.


## 2. Qu'est-ce qu'un équilibre corrélé ?

μ est une matrice de probabilités sur les paires d'actions (ici 2×2). Le médiateur
tire `(i,j) ~ μ` puis dit à M « joue `i` » et à J « joue `j` » (chacun ne voit que sa
propre recommandation).

μ est un **équilibre corrélé** si, pour chaque joueur, chaque recommandation, et
chaque action de déviation, *obéir* vaut au moins autant que *dévier* — **conditionnel
à la recommandation reçue** :

> pour tout `i, i'` (actions de M) :  Σⱼ μ(i,j) · [ A(i,j) − A(i',j) ] ≥ 0
> pour tout `j, j'` (actions de J) :  Σᵢ μ(i,j) · [ B(i,j) − B(i,j') ] ≥ 0

C'est un ensemble de **contraintes linéaires** en μ (plus Σμ = 1 et μ ≥ 0) → un
**polytope convexe**. `scipy.optimize.linprog` le manipule.


In [2]:
from scipy.optimize import linprog


def ce_constraints(A, B):
    """Inegalites de non-deviation de tout CE, en variables d = vec(mu)
    = [mu00, mu01, mu10, mu11]. Forme a.d <= b (pour linprog)."""
    m, n = A.shape
    ineq, const = [], []
    # Joueur M : pour chaque i, i' (i != i')
    for i in range(m):
        for ip in range(m):
            if i == ip:
                continue
            row = np.zeros(m * n)
            for j in range(n):
                row[i * n + j] = A[i, j] - A[ip, j]
            ineq.append(-row)
            const.append(0.0)
    # Joueur J : pour chaque j, j' (j != j')
    for j in range(n):
        for jp in range(n):
            if j == jp:
                continue
            row = np.zeros(m * n)
            for i in range(m):
                row[i * n + j] = B[i, j] - B[i, jp]
            ineq.append(-row)
            const.append(0.0)
    return np.array(ineq), np.array(const)


def all_ce_lp(A, B, objective=None):
    """LP complet des CE : non-deviation + sum(mu)=1 + mu>=0.
    `objective` (vec m*n) = payoff a MAXIMISER ; None = simple feasibility."""
    m, n = A.shape
    ineq, const = ce_constraints(A, B)
    c = np.zeros(m * n) if objective is None else -objective
    return linprog(c, A_ub=ineq, b_ub=const,
                   A_eq=np.ones((1, m * n)), b_eq=[1.0],
                   bounds=[(0, None)] * (m * n), method="highs")


def is_ce(mu, A, B, tol=1e-8):
    """Vrai si mu est un equilibre corrélé (toutes non-déviations satisfaites)."""
    ineq, _ = ce_constraints(A, B)
    d = np.asarray(mu, dtype=float).reshape(-1)
    return bool(np.all(ineq @ d <= tol)) and abs(d.sum() - 1) < tol and (d >= -tol).all()


def expected_payoff(mu, A, B):
    """Payoff en attente (M, J) sous la distribution jointe mu."""
    return (mu * A).sum(), (mu * B).sum()


ineq, const = ce_constraints(A, B)
print("Inegalites de non-deviation :", ineq.shape[0],
      "  (M : F->O, O->F   ;  J : F->O, O->F)")


Inegalites de non-deviation : 4   (M : F->O, O->F   ;  J : F->O, O->F)


### Lecture du résultat — quatre inégalités, deux par joueur

La sortie précédente affiche `Inegalites de non-deviation : 4  (M : F->O, O->F ;
J : F->O, O->F)`. Le compte est **structurel** : l'inégalité est indexée par le couple
(recommandation reçue, action déviée). Chaque joueur a 2 recommandations possibles et,
pour chacune, une seule autre action vers laquelle dévier — soit deux inégalités par
joueur, donc **quatre** au total.

Sur ce jeu précis, elles s'écrivent : pour M, `3·μ₀₀ ≥ μ₀₁` (recommandé F, ne pas
basculer sur O) et `μ₁₁ ≥ 3·μ₁₀` (recommandé O, ne pas dévier vers F) ; pour J,
`μ₀₀ ≥ 3·μ₁₀` (recommandé F, ne pas dévier vers O) et `3·μ₁₁ ≥ μ₀₁` (recommandé O, ne
pas dévier vers F). Chacune compare le gain d'**obéir** au gain de **dévier**, à
recommandation donnée : c'est la condition centrale de la définition.

Les quatre briques sont donc en place : `ce_constraints` (les inégalités),
`all_ce_lp` (le solveur sur le polytope), `is_ce` (le test d'appartenance) et
`expected_payoff` (le payoff en attente). Les deux dernières servent de **témoins
indépendants** : `is_ce` ne réutilise pas le solveur, il vérifie les inégalités
directement — ce qui permet de contrôler le LP au lieu de le croire sur parole.


In [3]:
# CE a payoff social maximal : objective = sum_ij mu_ij (A_ij + B_ij)
welfare = (A + B).reshape(-1)
res = all_ce_lp(A, B, objective=welfare)
mu_ce = res.x.reshape(A.shape)
print("Statut LP :", res.status, "-", res.message)
print("CE a payoff social maximal :", np.round(mu_ce, 3).tolist())
print("  support :", [(actions[i], actions[j], round(float(mu_ce[i, j]), 3))
                     for i in range(2) for j in range(2) if mu_ce[i, j] > 1e-6])
print("  payoff social maximal :", round(float(-res.fun), 3))


Statut LP : 0 - Optimization terminated successfully. (HiGHS Status 7: Optimal)
CE a payoff social maximal : [[1.0, 0.0], [0.0, 0.0]]
  support : [('F', 'F', 1.0)]
  payoff social maximal : 4.0


### Lecture du résultat — pourquoi 4, et pourquoi le LP a rendu un coin

La sortie affiche `Statut LP : 0 - Optimization terminated successfully. (HiGHS Status
7: Optimal)`, `CE a payoff social maximal : [[1.0, 0.0], [0.0, 0.0]]`, un support
réduit à `('F', 'F', 1.0)` et un `payoff social maximal : 4.0`.

Le maximum se **comprend** sans solveur : le payoff social vaut `A[i,j] + B[i,j]`,
donc 4 sur les deux coins de la diagonale et 0 sur les deux cases hors diagonale — soit
`4·(μ₀₀ + μ₁₁)`, au plus 4. L'ensemble qui atteint 4 est donc exactement celui des
distributions qui **portent toute leur masse sur la diagonale** : un segment de (F,F)
à (O,O), et non un point unique.

Le LP a pourtant rendu le **coin** (F,F), pas le milieu de ce segment : sur une face où
l'objectif est constant, tout point est optimal, et un solveur de LP rend un **point
extrême**. Les deux coins de la diagonale sont donc optimaux au même titre que le
médiateur du paragraphe suivant — l'unicité de la sortie du solveur est un artefact de
l'algorithme, pas un fait économique. C'est pourquoi le carnet mesure le médiateur
**explicitement**, au lieu de lire la solution du LP.


In [4]:
# Le mediateur equitable : 1/2 en (F,F), 1/2 en (O,O)
mu_mediator = np.array([[0.5, 0.0],
                        [0.0, 0.5]])
print("Le mediateur (1/2 F-F, 1/2 O-O) est un CE ?", is_ce(mu_mediator, A, B))
pM, pJ = expected_payoff(mu_mediator, A, B)
print(f"  -> payoff : M={pM:.3f}, J={pJ:.3f}   (payoff social {pM + pJ:.3f})")


Le mediateur (1/2 F-F, 1/2 O-O) est un CE ? True
  -> payoff : M=2.000, J=2.000   (payoff social 4.000)


### Lecture du résultat — le médiateur juste et son payoff

La sortie affiche `Le mediateur (1/2 F-F, 1/2 O-O) est un CE ? True`, puis
`M=2.000, J=2.000 (payoff social 4.000)`.

Deux lectures se superposent. La première est **sociale** : le payoff social est
4.000, soit le **maximum** mesuré au paragraphe précédent — le médiateur ne perd rien
par rapport à la meilleure issue atteignable, il atteint le sommet de la face
optimale. La seconde est **individuelle** : chacun obtient 2.000, c'est-à-dire
exactement la moitié du surplus, là où les deux équilibres purs se partageaient 3 et 1.

Le médiateur est donc **équitable** au sens fort : il donne aux deux joueurs la même
part du surplus maximal, sans qu'aucun ait à être le « loser » de la coordination.
Reste à comparer ce 2.000 au Nash du même jeu — c'est l'objet du §3, où le Nash mixte
est calculé pour être mesuré dans les mêmes unités.


## 3. Le Nash, mesuré — pour voir l'écart

On calcule le Nash **mixte** du même jeu (les purs sont les deux coins). L'écart
avec le CE n'est *mesurable* que si le Nash est mesuré de la même façon — d'où le
passage par `expected_payoff` sur la distribution jointe du Nash mixte.


In [5]:
# Nash mixte de BoS. Le jeu est ASYMETRIQUE : les deux seuils d'indifference
# sont DIFFERENTS (pas un seul q commun, comme pour un jeu symetrique).
#   M indifferant entre F et O :  3*q_J = 1*(1-q_J)   =>  q_J = 1/4
#   J indifferant entre F et O :  1*p_M = 3*(1-p_M)   =>  p_M = 3/4
pM_F = 3 / 4   # M joue F avec 3/4
pJ_F = 1 / 4   # J joue F avec 1/4
pM_dist = np.array([pM_F, 1 - pM_F])
pJ_dist = np.array([pJ_F, 1 - pJ_F])
mu_nash_mixed = np.outer(pM_dist, pJ_dist)
pM_nash, pJ_nash = expected_payoff(mu_nash_mixed, A, B)
print(f"Nash mixte : M joue F avec {pM_F:.3f}, J joue F avec {pJ_F:.3f}")
print(f"  -> M={pM_nash:.3f}, J={pJ_nash:.3f}")
print(f"  distribution jointe : {mu_nash_mixed.tolist()}")
print("  Nash purs : (F,F) -> (3,1)    et    (O,O) -> (1,3)")


Nash mixte : M joue F avec 0.750, J joue F avec 0.250
  -> M=0.750, J=0.750
  distribution jointe : [[0.1875, 0.5625], [0.0625, 0.1875]]
  Nash purs : (F,F) -> (3,1)    et    (O,O) -> (1,3)


### Interprétation — le tableau de bord

| Équilibre          | Payoff M | Payoff J |
|--------------------|----------|----------|
| Nash pur (F,F)     | 3.0      | 1.0      |
| Nash pur (O,O)     | 1.0      | 3.0      |
| Nash mixte         | 0.75     | 0.75     |
| **CE médiateur**   | **2.0**  | **2.0**  |

Le CE médiateur **(2;2)** domine **strictement** le Nash mixte **(0.75;0.75)** au sens
de Pareto — il améliore les deux joueurs à la fois. C'est le résultat central d'Aumann :
**la corrélation coordonnée par un médiateur atteint un accord que les joueurs ne
trouvent pas seuls** — et l'écart est d'autant plus net que le Nash mixte, lui,
égalise déjà les parts sans pouvoir les monter.
Le tableau se lit **colonne par colonne** : ce qu'il faut comparer, c'est M et J
séparément. Le CE médiateur gagne **2.0** là où le Nash mixte gagne **0.75** — soit
1.25 de plus pour chacun : le médiateur domine le Nash mixte dans **les deux**
colonnes à la fois (domination de Pareto stricte). Les deux Nash purs, eux, ne sont
**pas** dominés par le médiateur : (F,F) donne 3.0 à M contre 2.0 au médiateur. Ce que
le médiateur domine, c'est donc le Nash **mixte**, et c'est précisément là que se situe
le manque : un équilibre de Nash **égalise** les parts (0.75 ; 0.75) au lieu de les
**maximiser**, parce que personne n'a de dispositif pour recommander.


## 4. Où le Nash vit dans le polytope — et où il ne vit pas

Deux faits à **mesurer**, qui tracent la frontière exacte entre Nash et corrélé :

- les **Nash purs** (F,F) et (O,O) : des CE « dégénérés » (masse ponctuelle) — ils
  forment des **sommets** du polytope des équilibres corrélés ;
- le **Nash mixte**, vu comme distribution *indépendante* (produit des stratégies) :
  il est **lui aussi** un équilibre corrélé. Ce n'est pas un hasard : c'est le contenu
  du théorème d'Aumann — **tout équilibre de Nash est un équilibre corrélé** — que le
  carnet va *mesurer* avec le même `is_ce`.


In [6]:
# Les deux Nash purs sont des CE (les sommets du polytope) :
for (i, j, lab) in [(0, 0, "(F,F)"), (1, 1, "(O,O)")]:
    mu = np.zeros((2, 2)); mu[i, j] = 1.0
    print(f"Nash pur {lab} est un CE ?", is_ce(mu, A, B))

# Le Nash mixte EST un CE : tout equilibre de Nash est un equilibre correle (Aumann 1974).
# C'est le theoreme que ce carnet presente -- mesure ici sur la distribution du §3.
print("Nash mixte est un CE ?", is_ce(mu_nash_mixed, A, B),
      "   <- True : le Nash est INCLUS dans le corrélé")

# Controle negatif : la distribution UNIFORME sur les 4 profils est corrélée mais n'est
# PAS un CE -- la corrélation seule ne suffit pas, il faut la non-déviabilité conditionnelle.
mu_uniform = np.full((2, 2), 0.25)
print("Distribution uniforme est un CE ?", is_ce(mu_uniform, A, B),
      "   <- False : le polytope est strictement plus petit que le simplexe")


Nash pur (F,F) est un CE ? True
Nash pur (O,O) est un CE ? True
Nash mixte est un CE ? True    <- True : le Nash est INCLUS dans le corrélé
Distribution uniforme est un CE ? False    <- False : le polytope est strictement plus petit que le simplexe


### Lecture

La frontière est claire et **mesurée** :
- les **Nash purs** sont des CE — les sommets du polytope ;
- le **Nash mixte** est un CE aussi : c'est le théorème d'Aumann, « tout équilibre de
  Nash est un équilibre corrélé », vérifié par le même `is_ce` ;
- la distribution **uniforme**, elle, n'en est **pas** un — la corrélation seule ne
  fait pas un équilibre, il faut en plus la non-déviabilité conditionnelle. Le polytope
  des équilibres corrélés est donc **strictement plus petit** que le simplexe.

C'est là que la corrélation gagne : le médiateur **(2;2)** est un CE que **aucun**
équilibre de Nash n'atteint (ni les purs, ni le mixte à 0.75;0.75), et **aucun** CE
n'atteint (3;3). La corrélation n'est pas un détail : c'est ce qui sépare le médiateur
(2;2) du meilleur Nash (0.75;0.75).
Le contre-exemple uniforme mérite d'être lu de près : c'est **aussi** une distribution
— donc tout aussi « corrélé » au sens courant du mot — et pourtant `is_ce` le refuse.
Ce qui sépare les deux, c'est la **conditionnalité** : la non-déviabilité ne se lit pas
sur la distribution prise globalement, mais sur la distribution **conditionnée à la
recommandation reçue**. La distribution uniforme sur les quatre profils ne concentre
aucune masse sur les profils où dévier serait tentant : elle ne dit rien de ce que
ferait un joueur *sachant* qu'on lui recommande F.

Ce que le carnet a ainsi **établi** sur le polytope : il contient les deux Nash purs,
le Nash mixte et le médiateur, et la distribution uniforme n'y figure pas. Quelle est sa
forme exacte reste à établir : la section
suivante en donne la méthode.


## 5. Exercices

Trois exercices pour ancrer le mécanisme. Chaque fonction est un **stub** à compléter
(`return None`) ; le carnet s'exécute de bout en bout même incomplet.

### Exercice 1 — Vérifieur de CE

**Objectif** : écrire un test d'appartenance au polytope des équilibres corrélés,
indépendant du solveur LP.
**Indice** : `ce_constraints(A, B)` renvoie les inégalités ; il suffit de tester
`ineq @ mu.reshape(-1) <= 0` sur chacune.
**Vérification attendue** : `is_ce_custom(mu_mediator, A, B)` vaut `True`, et
`is_ce_custom(mu_uniform, A, B)` vaut `False` — le même couple de témoins que la
cellule de contrôle négatif.

### Exercice 2 — CE à payoff maximal pour un AUTRE jeu

**Objectif** : refaire le calcul sur un jeu de coordination différent, pour vérifier que
la méthode ne dépend pas des gains de Battle of the Sexes.
**Indice** : l'objectif du LP est `objective = (A2 + B2).reshape(-1)`, puis
`all_ce_lp(A2, B2, objective=...)`.
**Vérification attendue** : `max_welfare_ce(A2, B2)` renvoie une matrice 2×2 dont le
support est porté par les deux coins de la diagonale — les gains sociaux y sont égaux
(4 + 2 et 2 + 4), ce qui reproduit la face optimale du carnet.

### Exercice 3 — Lequel gagne ?

**Objectif** : formuler l'écart entre l'issue corrélée et l'issue de Nash, en une phrase
qui cite les deux payoffs.
**Indice** : `expected_payoff(mu_mediator, A, B)` donne (2.0, 2.0) ;
`(pM_nash, pJ_nash)` donne (0.75, 0.75).
**Vérification attendue** : la phrase doit nommer les deux couples de payoffs et
conclure que le CE domine le Nash mixte **pour les deux joueurs** (domination de Pareto
stricte).

### Pour aller plus loin — la forme exacte du polytope

Le carnet a établi que le polytope contient (F,F), (O,O), le Nash mixte et le
médiateur, et que la distribution uniforme n'y figure pas. **Question** : combien de
sommets a-t-il, et quelles sont leurs coordonnées ?

**Piste** : un sommet du polytope est un point que les contraintes actives suffisent à
déterminer — par exemple deux inégalités de non-déviabilité, complétées par les
non-négativités μ ≥ 0. On peut énumérer les combinaisons de contraintes actives et
résoudre les systèmes linéaires correspondants, ou bien échantillonner le polytope par
LP (optimiser chaque coordonnée de μ dans les deux sens pour enrober ses extrêmes).
**Piège à éviter** : conclure d'un point échantillonné qu'il s'agit d'un sommet — un
point échantillonné n'en est un que si aucune direction ne permet de s'en éloigner tout
en restant dans le polytope.


In [7]:
# Exercice 1 — Vérifieur de CE
# Compléter is_ce_custom : renvoyer True si mu est un équilibre corrélé.
# Indice : réutiliser ce_constraints(A, B) et tester  ineq @ mu.reshape(-1) <= 0.
def is_ce_custom(mu, A, B):
    # TODO etudiant
    return None
# Test attendu : is_ce_custom(mu_mediator, A, B) == True


In [8]:
# Exercice 2 — CE à payoff maximal pour un AUTRE jeu
# Soit le jeu de coordination ci-dessous. Compléter max_welfare_ce pour renvoyer la
# distribution mu (matrice 2x2) du CE à payoff social maximal.
# Indice : objective = (A2 + B2).reshape(-1), puis all_ce_lp(A2, B2, objective=...).
A2 = np.array([[4, 0], [0, 2]])
B2 = np.array([[2, 0], [0, 4]])


def max_welfare_ce(A2, B2):
    # TODO etudiant
    return None
# mu_etoile = max_welfare_ce(A2, B2)  -> support attendu sur les deux coins


In [9]:
# Exercice 3 — Lequel gagne ?
# Pour le jeu de base (A, B) : compléter ce_verdict pour renvoyer une phrase
# comparant le payoff du CE médiateur à celui du Nash mixte.
# Indice : expected_payoff(mu_mediator, A, B)  vs  (pM_nash, pJ_nash).
def ce_verdict(mu_mediator, A, B, pM_nash, pJ_nash):
    # TODO etudiant — ex : "Le CE mediateur (2.0; 2.0) domine le Nash mixte
    #                         (0.75; 0.75) pour J et ameliore M."
    return None


## À retenir

- L'équilibre **corrélé** est une distribution μ sur les profils d'actions que aucun
  joueur ne veut « trahir » sachant sa recommandation — un **polytope LP**.
- Sur un jeu de coordination, le médiateur atteint **(2;2)** là où le Nash mixte
  plafonne à **(0.75;0.75)** : la corrélation **crée** de la valeur.
- Le Nash est **inclus** dans le corrélé — purs et mixte y sont des CE, ce que `is_ce`
  mesure : c'est le théorème d'Aumann, un raffinement et non un remplacement.
- Le polytope est **strictement plus petit** que le simplexe : la distribution uniforme,
  corrélée, n'est pas un équilibre (mesuré) — la corrélation nécessite la
  non-déviabilité conditionnelle.

**Prochaine étape de la série** : les **jeux répétés**
([Dynamics-01 — Jeux répétés et théorème Folk](Dynamics/Dynamics-01-Jeux-Repetes-Folk-Theorem-Python.ipynb)),
où l'historique remplace la recommandation : la coordination se construit par des
stratégies conditionnées à ce qui s'est passé, et non par un tirage unique observé
une fois pour toutes.


## Vérifiez votre compréhension

1. Le Nash mixte donne `(0.750, 0.750)` à chacun, tandis que `(F, F)` donne `(3, 1)` et `(O, O)` donne `(1, 3)`. Ces trois équilibres sont-ils comparables ?

   <details><summary>Réponse</summary>

   Non, et c'est le point le plus instructif du carnet : le Nash mixte est **Pareto-dominé** par les deux équilibria purs (M passe de `0.750` à `3` ou à `1`, J de `0.750` à `1` ou à `3`), et pourtant il reste un équilibre de Nash. Au profil mixte, M est indifférent (`0.750` qu'il joue F ou O, le joueur adverse jouant F avec probabilité 0.250) et J l'est aussi. Un équilibre de Nash n'est pas un optimum de Pareto : c'est le point d'appui de tout le carnet.

   </details>

2. Le médiateur est un équilibre corrélé, avec `M=2.000, J=2.000` (bien-être social `4.000`), là où le Nash mixte plafonne à `(0.750, 0.750)`. Qu'est-ce que la corrélation crée, exactement ?

   <details><summary>Réponse</summary>

   De la **valeur**, en supprimant la loterie : sous une recommandation commune, le joueur sait ce qu'il joue et le jeu devient déterministe pour lui. Le médiateur (1/2 F-F, 1/2 O-O) atteint un bien-être social de `4.000`, contre `1.500` pour le Nash mixte. La recommandation ne fait pas que rééquilibrer les payoffs : elle change le montant total dont dispose le collectif.

   </details>

3. La distribution uniforme n'est pas un équilibre corrélé, et le LP du bien-être social maximal renvoie le profil pur `(F, F)` avec un bien-être de `4.0`. Que prouve le contrôle négatif ?

   <details><summary>Réponse</summary>

   Que le polytope est **strictement plus petit** que le simplexe : la non-déviabilité conditionnelle contraint réellement la distribution, et pas seulement son support. Si le polytope était tout le simplexe, toute distribution — l'uniforme comprise — serait un CE. C'est exactement le contrôle qu'un vérificateur honnête doit porter : un test qui n'a que des cas positifs ne prouve rien.

   </details>

4. Le Nash pur `(F, F)` est vérifié CE, de même que `(O, O)` et le Nash mixte. Quelle relation entre Nash et corrélé cela établit-il — et que reste-t-il à établir ?

   <details><summary>Réponse</summary>

   Cela établit le **contenu** du théorème d'Aumann : tout équilibre de Nash est un équilibre corrélé — le Nash est un **raffinement** du corrélé, non un remplacement, et le carnet le dit dans les deux sens (« le Nash est INCLUS dans le corrélé »). Ce qui reste, et que le carnet mesure, est le versant inverse : le corrélé autorise des profils (le médiateur) qui ne sont pas des Nash, et c'est là que se crée la valeur.

   </details>


## Pour aller plus loin

**Carnet suivant.** [GameTheory-05 — Zero-sum et minimax](GameTheory-05-ZeroSum-Minimax-Python.ipynb) -- le palier suivant de la série : là où le carnet présent sortait de la famille 04 par la recommandation commune, le palier 05 change de nature de jeu (somme nulle) et de solution (minimax).

**Lectures.**

- La question laissée ouverte par le carnet — le nombre de sommets du polytope et leurs coordonnées — se trouve en fin de la section « 5. Exercices » (« La forme exacte du polytope ») : elle demande d'énumérer les combinaisons de contraintes actives, ou d'enrober le polytope par LP, en prenant garde au piège d'un point échantillonné confondu avec un sommet.
- Les références complètes du carnet sont dans la section « Références » : Aumann (1974), l'article fondateur qui définit l'équilibre corrélé et établit le résultat que ce carnet mesure (tout équilibre de Nash est un équilibre corrélé), avec l'archivage et l'empreinte de la copie consultée.
- Le prolongement de la série au-delà de la famille 04 est les **jeux répétés** ([Dynamics-01 — Jeux répétés et théorème Folk](Dynamics/Dynamics-01-Jeux-Repetes-Folk-Theorem-Python.ipynb)) : là où le médiateur remplace l'histoire, l'historique construit la coordination par des stratégies conditionnées à ce qui s'est passé.

**Carnets voisins.**

- [GameTheory-04 — Nash](GameTheory-04-NashEquilibrium-Python.ipynb) -- le track principal du palier, dont ce carnet est l'accretion sur la corrélation.
- [GameTheory-04j — Rationalisabilité](GameTheory-04j-Rationalisabilite-Python.ipynb) -- ce que la seule rationalité délie, quand la cohérence mutuelle faiblit.
- [GameTheory-05 — Zero-sum et minimax](GameTheory-05-ZeroSum-Minimax-Python.ipynb) -- le palier suivant de la série.


## Références

* **Aumann, R. J. (1974).** *Subjectivity and correlation in randomized strategies.*
  *Journal of Mathematical Economics*, 1(1), 67–96. — l'article fondateur qui définit
  l'équilibre corrélé et établit le résultat que ce carnet mesure : tout équilibre de
  Nash est un équilibre corrélé. Édition Elsevier (DOI `10.1016/0304-4068(74)90037-8`) :
  <https://www.sciencedirect.com/science/article/pii/0304406874900378>.

  *Source archivée* : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\Bayesian Games, Information and Correlation\1974 - Aumann - Subjectivity and Correlation in Randomized Strategies (JME 1(1) 67-96, reprint ch.31 p.559-592).pdf`
  (sha8 `E3D64416`, 33 pages). Identité vérifiée sur la **première page** du
  reprint, qui porte la note : « This chapter originally appeared in Journal of
  Mathematical Economics 1 (1974): 67–96. Reprinted with permission. » — le PDF
  est hébergé sur la page institutionnelle de l'auteur (HUJI).
